# US Option Match — ADR universe screen

Decides which Taiwan stocks are eligible for the **US Option Match** tab. A
candidate is admitted only if it clears **all** gates:

1. **Has a US ADR** on a US exchange, with a known ADR ratio (ordinary shares per ADR).
2. **ADR ≈ same asset as local share** — after normalizing by ratio + FX, the
   premium is small and rarely diverges (reject if `|premium| > 5%` happens too
   often, or the mean premium is large/persistent).
3. **Warrants trade in Taiwan** on the underlying.
4. **Options trade on the US ADR** (has expiries).

Same normalization the app uses:

    adr_implied_twd_per_share = adr_usd / adr_ratio * FX   (FX = TWD per USD)

In [1]:
import yfinance as yf, pandas as pd, datetime, twstock

def daily(t, p="3y"):
    s = yf.Ticker(t).history(period=p)["Close"]
    s.index = pd.to_datetime(s.index.date)
    return s[~s.index.duplicated()]

fx = daily("TWD=X")
today = datetime.datetime.today()

# candidate: adr_ticker -> (taiwan_code, taiwan_name, assumed adr_ratio)
candidates = {
    "TSM": ("2330", "台積電", 5),
    "UMC": ("2303", "聯電", 5),
    "ASX": ("3711", "日月光投控", 2),
    "CHT": ("2412", "中華電", 10),
}

def warrant_count(code, name):
    return sum(
        1 for k, v in twstock.codes.items()
        if "權證" in v.type
        and (name in v.name or v.name.startswith(code))
        and datetime.datetime.strptime(v.start, "%Y/%m/%d") <= today
    )

rows = []
for adr, (code, name, ratio) in candidates.items():
    a, t = daily(adr), daily(f"{code}.TW")
    df = pd.DataFrame({"a": a, "t": t})
    df["fx"] = fx.reindex(df.index).ffill()
    df = df.dropna()
    impl_ratio = (df["a"] * df["fx"] / df["t"]).median()
    df["adr_tw"] = df["a"] / ratio * df["fx"]
    df["prem"] = (df["adr_tw"] / df["t"] - 1) * 100
    ret = df[["t", "adr_tw"]].pct_change().dropna()
    rows.append({
        "adr": adr, "tw": code, "ratio": ratio,
        "impl_ratio": round(impl_ratio, 2),
        "prem_mean%": round(df["prem"].mean(), 2),
        "prem_std": round(df["prem"].std(), 2),
        "|prem|>5%": round((df["prem"].abs() > 5).mean() * 100, 1),
        "ret_corr": round(ret["t"].corr(ret["adr_tw"]), 2),
        "us_expiries": len(yf.Ticker(adr).options),
        "tw_warrants": warrant_count(code, name),
    })

screen = pd.DataFrame(rows).set_index("adr")
screen

,tw,ratio,impl_ratio,prem_mean%,prem_std,|prem|>5%,ret_corr,us_expiries,tw_warrants
adr,,,,,,,,,
TSM,2330,5,5.88,17.50,6.04,99.4,0.31,17,1265
UMC,2303,5,5.00,0.15,2.00,2.6,0.58,5,269
ASX,3711,2,2.10,5.80,4.79,48.7,0.46,6,0
CHT,2412,10,9.99,-0.05,0.79,0.1,0.49,2,4


In [2]:
# Verdict: admit only if parity tight AND warrants exist AND options exist.
def verdict(r):
    if r["tw_warrants"] == 0:      return "REJECT — no TW warrants"
    if r["us_expiries"] == 0:      return "REJECT — no US options"
    if abs(r["prem_mean%"]) > 3 or r["|prem|>5%"] > 10:
        return "REJECT — ADR premium too large/frequent"
    return "ADMIT"

screen["verdict"] = screen.apply(verdict, axis=1)
print(screen[["impl_ratio", "prem_mean%", "|prem|>5%", "ret_corr",
              "us_expiries", "tw_warrants", "verdict"]].to_string())
print("\nAdmitted:", list(screen[screen.verdict == "ADMIT"].index))

     impl_ratio  prem_mean%  |prem|>5%  ret_corr  us_expiries  tw_warrants                                  verdict
adr                                                                                                                
TSM        5.88       17.50       99.4      0.31           17         1265  REJECT — ADR premium too large/frequent
UMC        5.00        0.15        2.6      0.58            5          269                                    ADMIT
ASX        2.10        5.80       48.7      0.46            6            0                  REJECT — no TW warrants
CHT        9.99       -0.05        0.1      0.49            2            4                                    ADMIT

Admitted: ['UMC', 'CHT']
